# Import all libraries needed

In [ ]:
#data
import pickle
import pandas as pd
import numpy as np
import random

#deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

#evaluate
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

#system
import os
from tqdm import tqdm


# Read and preprocess data

In [ ]:
#read & preprocess data

##read data
with open('../../data/train.pickle', 'rb') as f:
    train_data = pickle.load(f)

train_df = pd.DataFrame(train_data)

def convert_label_to_binary(label):
    if pd.isna(label):
        return [0, 0, 0, 0]
    
    binary_mapping = {
        0: [0, 0, 0, 0],
        1: [0, 0, 1, 0],
        2: [0, 0, 0, 1],
        3: [0, 0, 1, 1],
        4: [1, 0, 0, 0],
        5: [0, 1, 0, 0],
        6: [1, 1, 0, 0],
        7: [1, 0, 1, 0],
        8: [0, 1, 0, 1],
        9: [1, 0, 0, 1],
        10: [0, 1, 1, 0],
        11: [1, 1, 1, 1]
    }
    return binary_mapping[label]

def binary_to_original_label(binary_labels):
    label_mapping = {
        (0, 0, 0, 0): 0,
        (0, 0, 1, 0): 1,
        (0, 0, 0, 1): 2,
        (0, 0, 1, 1): 3,
        (1, 0, 0, 0): 4,
        (0, 1, 0, 0): 5,
        (1, 1, 0, 0): 6,
        (1, 0, 1, 0): 7,
        (0, 1, 0, 1): 8,
        (1, 0, 0, 1): 9,
        (0, 1, 1, 0): 10,
        (1, 1, 1, 1): 11
    }
    
    original_labels = []
    for binary in binary_labels:
        binary_tuple = tuple(map(int, binary))
        original_labels.append(label_mapping.get(binary_tuple, -1))
    
    return original_labels

##create sub label
binary_labels = train_df['label'].apply(convert_label_to_binary)

train_df[['label_1', 'label_2', 'label_3', 'label_4']] = pd.DataFrame(binary_labels.tolist())

# Deep learning model definition of GAN and RNN
**Also find possibilities wtih architecture of cGAN, DANN, Transformer**

In [ ]:
#deep learning model arch

class ScaledTanh(nn.Module):
    def __init__(self, scale=5.0):
        super(ScaledTanh, self).__init__()
        self.scale = scale
        
    def forward(self, x):
        return torch.tanh(x) * self.scale

##GAN

###Generator
class Generator(nn.Module):
    def __init__(self, latent_dim=100, sensor_seq_len=128, sensor_channels=6, num_labels=4):
        super().__init__()
        self.latent_dim = latent_dim
        self.sensor_seq_len = sensor_seq_len
        self.sensor_channels = sensor_channels
        self.num_labels = num_labels
        
        # Network to generate sensor data
        self.sensor_generator = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 1536),
            nn.BatchNorm1d(1536),
            nn.LeakyReLU(0.2),
            nn.Linear(1536, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(0.2),
            nn.Linear(2048, sensor_seq_len * sensor_channels),
            ScaledTanh(scale=5.0)
        )
        
        # Network to generate labels
        self.label_generator = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, num_labels),
            nn.Sigmoid()
        )
        
    def forward(self, z):
        # Generate sensor data
        sensor_data = self.sensor_generator(z)
        sensor_data = sensor_data.view(-1, self.sensor_seq_len, self.sensor_channels)
        
        # Generate labels
        labels = self.label_generator(z)
        
        return sensor_data, labels

# Discriminator
class Discriminator(nn.Module):
    def __init__(self, sensor_seq_len=128, sensor_channels=6, num_labels=4):
        super().__init__()
        self.sensor_seq_len = sensor_seq_len
        self.sensor_channels = sensor_channels
        
        # Network to process sensor data
        self.sensor_processor = nn.Sequential(
            nn.Linear(sensor_seq_len * sensor_channels, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2)
        )
        
        # Combined discriminator network
        self.discriminator = nn.Sequential(
            nn.Linear(128 + num_labels, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, sensor_data, labels):
        # Flatten sensor data
        sensor_flat = sensor_data.view(sensor_data.size(0), -1)
        
        # Process sensor data
        sensor_features = self.sensor_processor(sensor_flat)
        
        # Combine sensor features and labels
        combined = torch.cat([sensor_features, labels], dim=1)
        
        # Discriminate real/fake
        output = self.discriminator(combined)
        return output

#RNN

class RNN(nn.Module):
    def __init__(self, sensor_channels=6, num_labels=4, hidden_dim=128, num_layers=5):
        super().__init__()
        self.sensor_channels = sensor_channels
        self.num_labels = num_labels
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        input_dim = sensor_channels + num_labels
        
        # Use simple RNN instead of LSTM
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True)
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, sensor_channels),
            ScaledTanh(scale=5.0)
        )
        
    def forward(self, sensor_sequence, labels, teacher_forcing=True):
        
        batch_size, seq_len, _ = sensor_sequence.size()
        device = sensor_sequence.device
        
        # Expand labels to match sequence length
        expanded_labels = labels.unsqueeze(1).expand(-1, seq_len, -1)
        
        if teacher_forcing:
            # Use teacher forcing: input is the real previous values
            rnn_input = torch.cat([sensor_sequence, expanded_labels], dim=2)
            rnn_out, _ = self.rnn(rnn_input)
            output = self.output_layer(rnn_out)
            return output
        else:
            # Autoregressive generation
            return self.generate_sequence(sensor_sequence[:, 0, :], labels, seq_len)
    
    def generate_sequence(self, initial_sensor, labels, seq_len):
        """Generate sequence autoregressively"""
        batch_size = initial_sensor.size(0)
        device = initial_sensor.device
        
        # Initialize hidden state for RNN
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        
        generated_sequence = []
        current_sensor = initial_sensor.unsqueeze(1)
        h = h0
        
        for t in range(seq_len):
            current_labels = labels.unsqueeze(1)
            rnn_input = torch.cat([current_sensor, current_labels], dim=2)
            
            rnn_out, h = self.rnn(rnn_input, h)
            next_sensor = self.output_layer(rnn_out)
            
            generated_sequence.append(next_sensor)
            current_sensor = next_sensor
        
        return torch.cat(generated_sequence, dim=1)


##cGAN
###Conditional Generator
class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=100, condition_dim=4, sensor_seq_len=128, sensor_channels=6):
        super().__init__()
        self.latent_dim = latent_dim
        self.condition_dim = condition_dim
        self.sensor_seq_len = sensor_seq_len
        self.sensor_channels = sensor_channels
        
        # Condition embedding network
        self.condition_embedding = nn.Sequential(
            nn.Linear(condition_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2)
        )
        
        # Main generator network
        self.generator = nn.Sequential(
            nn.Linear(latent_dim + 256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(0.2),
            nn.Linear(2048, 1536),
            nn.BatchNorm1d(1536),
            nn.LeakyReLU(0.2),
            nn.Linear(1536, sensor_seq_len * sensor_channels),
            ScaledTanh(scale=5.0)
        )
        
    def forward(self, z, conditions):
        # Embed conditions
        condition_embedded = self.condition_embedding(conditions)
        
        # Concatenate noise and embedded conditions
        combined_input = torch.cat([z, condition_embedded], dim=1)
        
        # Generate sensor data
        sensor_data = self.generator(combined_input)
        sensor_data = sensor_data.view(-1, self.sensor_seq_len, self.sensor_channels)
        
        return sensor_data

###Conditional Discriminator
class ConditionalDiscriminator(nn.Module):
    def __init__(self, condition_dim=4, sensor_seq_len=128, sensor_channels=6):
        super().__init__()
        self.condition_dim = condition_dim
        self.sensor_seq_len = sensor_seq_len
        self.sensor_channels = sensor_channels
        
        # Condition embedding network
        self.condition_embedding = nn.Sequential(
            nn.Linear(condition_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2)
        )
        
        # Sensor data processing network
        self.sensor_processor = nn.Sequential(
            nn.Linear(sensor_seq_len * sensor_channels, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3)
        )
        
        # Combined discriminator network
        self.discriminator = nn.Sequential(
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
    def forward(self, sensor_data, conditions):
        # Flatten sensor data
        sensor_flat = sensor_data.view(sensor_data.size(0), -1)
        
        # Process sensor data
        sensor_features = self.sensor_processor(sensor_flat)
        
        # Embed conditions
        condition_embedded = self.condition_embedding(conditions)
        
        # Combine sensor features and conditions
        combined = torch.cat([sensor_features, condition_embedded], dim=1)
        
        # Discriminate real/fake
        output = self.discriminator(combined)
        return output

#Transformer GAN

class TransformerEncoder(nn.Module):
    def __init__(self, n_sensors=6, d_model=48, nhead=6, num_layers=3, seq_len=128):
        super().__init__()
        self.n_sensors = n_sensors
        self.d_model = d_model
        self.seq_len = seq_len
        
        self.sensor_embedding = nn.Linear(1, d_model // n_sensors)
        self.projection = nn.Linear((d_model // n_sensors) * n_sensors, d_model)
        self.position_embedding = nn.Parameter(torch.randn(1, seq_len, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dropout=0.2
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.flatten = nn.Flatten(1, 2)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, n_sensors = x.shape
        
        sensor_embeddings = []
        for i in range(self.n_sensors):
            sensor_data = x[:, :, i:i+1]
            embedded = self.sensor_embedding(sensor_data)
            sensor_embeddings.append(embedded)
        
        x = torch.cat(sensor_embeddings, dim=-1)
        x = self.projection(x)
        x = x + self.position_embedding[:, :seq_len, :]
        
        x_trans = x.transpose(1, 2)
        x_trans = self.batch_norm(x_trans)
        x = x_trans.transpose(1, 2)
        
        x = self.transformer(x, mask=mask)
        flattened_features = self.flatten(x)
        
        return flattened_features

# Generator with Transformer Encoder for sensor data processing
class TransformerGANGenerator(nn.Module):
    def __init__(self, latent_dim=100, d_model=48, num_labels=4, seq_len=128, n_sensors=6):
        super().__init__()
        self.latent_dim = latent_dim
        self.seq_len = seq_len
        self.n_sensors = n_sensors
        self.d_model = d_model
        
        # Calculate transformer output dimension
        self.transformer_output_dim = d_model * seq_len
        
        # Network to generate initial sensor data
        self.sensor_generator = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(True),
            nn.Linear(2048, seq_len * n_sensors),
            ScaledTanh(scale=5.0)
        )

        
        # Transformer encoder for processing generated sensor data
        self.transformer_encoder = TransformerEncoder(
            n_sensors=n_sensors, 
            d_model=d_model, 
            nhead=6, 
            num_layers=3, 
            seq_len=seq_len
        )
        
        # Network to generate labels based on transformer features and noise
        self.label_generator = nn.Sequential(
            nn.Linear(latent_dim + self.transformer_output_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.Linear(256, 128),
            nn.ReLU(True),
            nn.Linear(128, num_labels),
            nn.Sigmoid()
        )
        
    def forward(self, z):
        # Generate initial sensor data
        sensor_data = self.sensor_generator(z)
        sensor_data = sensor_data.view(-1, self.seq_len, self.n_sensors)
        
        # Process sensor data through transformer
        transformer_features = self.transformer_encoder(sensor_data)
        
        # Generate labels based on transformer features and noise
        combined_features = torch.cat([z, transformer_features], dim=1)
        labels = self.label_generator(combined_features)
        
        return sensor_data, labels

# Discriminator with Transformer Encoder for sensor data processing
class TransformerGANDiscriminator(nn.Module):
    def __init__(self, d_model=48, num_labels=4, seq_len=128, n_sensors=6):
        super().__init__()
        self.d_model = d_model
        self.transformer_output_dim = d_model * seq_len
        
        # Transformer encoder for processing sensor data
        self.transformer_encoder = TransformerEncoder(
            n_sensors=n_sensors, 
            d_model=d_model, 
            nhead=6, 
            num_layers=3, 
            seq_len=seq_len
        )
        
        # Discriminator network
        self.discriminator = nn.Sequential(
            nn.Linear(self.transformer_output_dim + num_labels, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, sensor_data, labels):
        # Process sensor data through transformer
        transformer_features = self.transformer_encoder(sensor_data)
        
        # Combine transformer features and labels
        combined = torch.cat([transformer_features, labels], dim=1)
        
        # Discriminate real/fake
        output = self.discriminator(combined)
        return output

#DANN
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()

    @staticmethod
    def backward(ctx, grads):
        lambda_ = ctx.lambda_
        lambda_ = grads.new_tensor(lambda_)
        dx = -lambda_ * grads
        return dx, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

    def set_lambda(self, lambda_):
        self.lambda_ = lambda_

# FCNN Feature Extractor
class FCNNFeatureExtractor(nn.Module):
    def __init__(self, sensor_seq_len=128, sensor_channels=6, feature_dim=512):
        super().__init__()
        input_dim = sensor_seq_len * sensor_channels
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(768, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
    def forward(self, x):
        # Flatten sensor data: (batch_size, seq_len, channels) -> (batch_size, seq_len * channels)
        x = x.view(x.size(0), -1)
        features = self.feature_extractor(x)
        return features

# Label Predictor
class LabelPredictor(nn.Module):
    def __init__(self, feature_dim=512, num_labels=4):
        super().__init__()
        self.predictor = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_labels),
            nn.Sigmoid()
        )
        
    def forward(self, features):
        return self.predictor(features)

# Domain Classifier
class DomainClassifier(nn.Module):
    def __init__(self, feature_dim=512, num_domains=2):
        super().__init__()
        self.grl = GradientReversalLayer()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_domains)
        )
        
    def forward(self, features):
        reversed_features = self.grl(features)
        domain_output = self.classifier(reversed_features)
        return domain_output
    
    def set_lambda(self, lambda_):
        self.grl.set_lambda(lambda_)

# DANN Model
class DANN(nn.Module):
    def __init__(self, sensor_seq_len=128, sensor_channels=6, feature_dim=512, 
                 num_labels=4, num_domains=2):
        super().__init__()
        self.feature_extractor = FCNNFeatureExtractor(sensor_seq_len, sensor_channels, feature_dim)
        self.label_predictor = LabelPredictor(feature_dim, num_labels)
        self.domain_classifier = DomainClassifier(feature_dim, num_domains)
        
    def forward(self, x):
        features = self.feature_extractor(x)
        label_output = self.label_predictor(features)
        domain_output = self.domain_classifier(features)
        return features, label_output, domain_output
    
    def set_lambda(self, lambda_):
        self.domain_classifier.set_lambda(lambda_)

# Domain-Invariant Generator
class DomainInvariantGenerator(nn.Module):
    def __init__(self, latent_dim=100, feature_dim=512, sensor_seq_len=128, 
                 sensor_channels=6, num_labels=4):
        super().__init__()
        self.latent_dim = latent_dim
        self.sensor_seq_len = sensor_seq_len
        self.sensor_channels = sensor_channels
        
        # Generator for sensor data
        self.sensor_generator = nn.Sequential(
            nn.Linear(latent_dim + feature_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(True),
            nn.Linear(1024, 1536),
            nn.BatchNorm1d(1536),
            nn.ReLU(True),
            nn.Linear(1536, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(True),
            nn.Linear(2048, sensor_seq_len * sensor_channels),
            ScaledTanh(scale=5.0)
        )
        
        # Generator for labels
        self.label_generator = nn.Sequential(
            nn.Linear(latent_dim + feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.Linear(256, num_labels),
            nn.Sigmoid()
        )
        
    def forward(self, z, domain_invariant_features):
        # Combine noise with domain-invariant features
        combined_input = torch.cat([z, domain_invariant_features], dim=1)
        
        # Generate sensor data
        sensor_data = self.sensor_generator(combined_input)
        sensor_data = sensor_data.view(-1, self.sensor_seq_len, self.sensor_channels)
        
        # Generate labels
        labels = self.label_generator(combined_input)
        
        return sensor_data, labels

# Domain-Invariant Discriminator
class DomainInvariantDiscriminator(nn.Module):
    def __init__(self, sensor_seq_len=128, sensor_channels=6, num_labels=4):
        super().__init__()
        input_dim = sensor_seq_len * sensor_channels + num_labels
        
        self.discriminator = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, sensor_data, labels):
        # Flatten sensor data and concatenate with labels
        sensor_flat = sensor_data.view(sensor_data.size(0), -1)
        combined = torch.cat([sensor_flat, labels], dim=1)
        output = self.discriminator(combined)
        return output

# Training and saving models

In [ ]:
#train model & save model

##Dataset

class BrakeDataset(Dataset):
    def __init__(self, df, augment=False, return_labels=True, return_domain=False):
        self.sensor_data = torch.FloatTensor(np.stack(df['sensor_data'].values))
        self.augment = augment
        self.return_labels = return_labels
        self.return_domain = return_domain
        
        # Multi-label setup
        if return_labels:
            self.label_1 = torch.FloatTensor(df['label_1'].values.reshape(-1, 1))
            self.label_2 = torch.FloatTensor(df['label_2'].values.reshape(-1, 1))
            self.label_3 = torch.FloatTensor(df['label_3'].values.reshape(-1, 1))
            self.label_4 = torch.FloatTensor(df['label_4'].values.reshape(-1, 1))
            self.all_labels = torch.cat([self.label_1, self.label_2, self.label_3, self.label_4], dim=1)
        
        # Domain labels
        if return_domain:
            unique_domains = df['model'].unique()
            self.domain_map = {domain: idx for idx, domain in enumerate(unique_domains)}
            self.domain_labels = torch.LongTensor([self.domain_map[model] for model in df['model'].values])
        
    def __len__(self):
        return len(self.sensor_data)
    
    def __getitem__(self, idx):
        sensor_data = self.sensor_data[idx]
        
        # Apply data augmentation if enabled
        if self.augment and random.random() > 0.7:
            sensor_data = self.augment_sensor_data(sensor_data)
        
        result = {'sensor_data': sensor_data}
        
        # Add labels if requested
        if self.return_labels:
            result['labels'] = self.all_labels[idx]
            result['label_1'] = self.label_1[idx]
            result['label_2'] = self.label_2[idx]
            result['label_3'] = self.label_3[idx]
            result['label_4'] = self.label_4[idx]
            
        # Add domain labels if requested
        if self.return_domain:
            result['domain_labels'] = self.domain_labels[idx]
            
        return result
    
    def augment_sensor_data(self, sensor_data, scale_factor=0.01):
        # Add small amount of Gaussian noise
        noise = torch.randn_like(sensor_data) * scale_factor
        return sensor_data + noise

class RNNDataset:
    def __init__(self, brake_dataset, sequence_length=128):
        self.brake_dataset = brake_dataset
        self.sequence_length = sequence_length
        
    def __len__(self):
        return len(self.brake_dataset)
    
    def __getitem__(self, idx):
        batch = self.brake_dataset[idx]
        sensor_data = batch['sensor_data']  # Shape: (128, 6)
        labels = batch['labels']  # Shape: (4,)
        
        # Create overlapping sequences for training
        total_length = sensor_data.shape[0]
        
        if total_length <= self.sequence_length:
            # If sequence is too short, pad or repeat
            input_seq = sensor_data[:-1] if total_length > 1 else sensor_data
            target_seq = sensor_data[1:] if total_length > 1 else sensor_data
        else:
            # Randomly sample a subsequence
            start_idx = torch.randint(0, total_length - self.sequence_length, (1,)).item()
            input_seq = sensor_data[start_idx:start_idx + self.sequence_length - 1]
            target_seq = sensor_data[start_idx + 1:start_idx + self.sequence_length]
        
        return {
            'input_sequence': input_seq,
            'target_sequence': target_seq,
            'labels': labels
        }

class UnifiedModelTrainer:
    def __init__(self, train_df, device='cuda', save_dir='../../results/models'):
        self.train_df = train_df
        self.device = device
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        
        # Create datasets
        self.dataset = BrakeDataset(train_df, augment=True, return_labels=True, return_domain=True)
        self.rnn_dataset = RNNDataset(self.dataset, sequence_length=128)
        
        # Initialize models
        self.models = {}
        self.optimizers = {}
        
        # Training history
        self.training_history = {}
        
    def initialize_models(self):
        """Initialize all models"""
        print("Initializing models...")
        
        # GAN
        self.models['gan_generator'] = Generator().to(self.device)
        self.models['gan_discriminator'] = Discriminator().to(self.device)

        # RNN
        self.models['rnn'] = RNN().to(self.device)
        
        # cGAN
        self.models['cgan_generator'] = ConditionalGenerator().to(self.device)
        self.models['cgan_discriminator'] = ConditionalDiscriminator().to(self.device)
        
        # Transformer GAN
        self.models['tgan_generator'] = TransformerGANGenerator().to(self.device)
        self.models['tgan_discriminator'] = TransformerGANDiscriminator().to(self.device)
        
        # DANN
        num_domains = len(self.train_df['model'].unique())
        self.models['dann'] = DANN(num_domains=num_domains).to(self.device)
        self.models['dann_generator'] = DomainInvariantGenerator().to(self.device)
        self.models['dann_discriminator'] = DomainInvariantDiscriminator().to(self.device)
        
        # Initialize optimizers
        self._initialize_optimizers()
        
    def _initialize_optimizers(self):
        """Initialize optimizers for all models with weight decay for regularization"""
        lr_gan = 0.005
        lr_rnn = 0.005
        lr_dann = 0.005
        weight_decay = 1e-4  # L2 regularization
        
        # GAN optimizers
        self.optimizers['gan_g'] = optim.Adam(self.models['gan_generator'].parameters(), 
                                            lr=lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        self.optimizers['gan_d'] = optim.Adam(self.models['gan_discriminator'].parameters(), 
                                            lr=0.2*lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)

        # RNN optimizer
        self.optimizers['rnn'] = optim.Adam(self.models['rnn'].parameters(), lr=lr_rnn, weight_decay=weight_decay)
        
        # cGAN optimizers
        self.optimizers['cgan_g'] = optim.Adam(self.models['cgan_generator'].parameters(), 
                                             lr=lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        self.optimizers['cgan_d'] = optim.Adam(self.models['cgan_discriminator'].parameters(), 
                                             lr=0.2*lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        
        # Transformer GAN optimizers
        self.optimizers['tgan_g'] = optim.Adam(self.models['tgan_generator'].parameters(), 
                                             lr=lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        self.optimizers['tgan_d'] = optim.Adam(self.models['tgan_discriminator'].parameters(), 
                                             lr=0.2*lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        
        # DANN optimizers
        self.optimizers['dann'] = optim.Adam(self.models['dann'].parameters(), lr=lr_dann, weight_decay=weight_decay)
        self.optimizers['dann_g'] = optim.Adam(self.models['dann_generator'].parameters(), 
                                             lr=lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
        self.optimizers['dann_d'] = optim.Adam(self.models['dann_discriminator'].parameters(), 
                                             lr=0.2*lr_gan, betas=(0.5, 0.999), weight_decay=weight_decay)
    
    def _add_dropout_regularization(self, model, dropout_rate=0.1):
        """Add dropout layers for regularization"""
        for module in model.modules():
            if isinstance(module, nn.Linear):
                # Add dropout after linear layers
                pass  # This would require modifying model architecture
    
    def train_all_models(self, epochs_dict=None):
        """Train all models"""
        if epochs_dict is None:
            epochs_dict = {
                'gan': 100,
                'cgan': 100,
                'rnn': 100,
                'tgan': 100,
                'dann': 100
            }
        
        print("Starting unified training process...")
        
        # Initialize training history
        for model_name in epochs_dict.keys():
            self.training_history[model_name] = {'epochs': [], 'losses': []}
        
        # Train GAN
        print("\n" + "="*50)
        print("Training GAN")
        print("="*50)
        self._train_gan(epochs_dict['gan'])

        # Train RNN
        print("\n" + "="*50)
        print("Training RNN")
        print("="*50)
        self._train_rnn(epochs_dict['rnn'])
        
        # Train cGAN
        print("\n" + "="*50)
        print("Training cGAN")
        print("="*50)
        self._train_cgan(epochs_dict['cgan'])
        
        # Train Transformer GAN
        print("\n" + "="*50)
        print("Training Transformer GAN")
        print("="*50)
        self._train_transformer_gan(epochs_dict['tgan'])
        
        # Train DANN
        print("\n" + "="*50)
        print("Training DANN")
        print("="*50)
        self._train_dann(epochs_dict['dann'])
        
        # Save all models
        self.save_all_models()
        
        # Print final training summary
        self._print_training_summary()
        
    def _train_gan(self, num_epochs):
        """Train GAN model with 1:5 discriminator:generator ratio"""
        dataloader = DataLoader(self.dataset, batch_size=64, shuffle=True)
        criterion = nn.BCELoss()
        
        epoch_losses_g = []
        epoch_losses_d = []
        
        for epoch in range(num_epochs):
            epoch_loss_g = 0
            epoch_loss_d = 0
            num_batches = 0
            
            for batch_idx, batch in enumerate(dataloader):
                real_sensor_data = batch['sensor_data'].to(self.device)
                real_labels = batch['labels'].to(self.device)
                batch_size = real_sensor_data.size(0)
                
                real_label = torch.ones(batch_size, 1).to(self.device) * 0.9
                fake_label = torch.zeros(batch_size, 1).to(self.device) + 0.1
                
                # Train Discriminator (every batch)
                self.optimizers['gan_d'].zero_grad()
                
                # Real data
                output_real = self.models['gan_discriminator'](real_sensor_data, real_labels)
                loss_D_real = criterion(output_real, real_label)
                
                # Fake data
                z = torch.randn(batch_size, 100).to(self.device)
                fake_sensor_data, fake_labels = self.models['gan_generator'](z)
                output_fake = self.models['gan_discriminator'](fake_sensor_data.detach(), fake_labels.detach())
                loss_D_fake = criterion(output_fake, fake_label)
                
                # L1 regularization for discriminator
                l1_reg_d = 0
                for param in self.models['gan_discriminator'].parameters():
                    l1_reg_d += torch.sum(torch.abs(param))
                
                loss_D = loss_D_real + loss_D_fake + 1e-5 * l1_reg_d
                loss_D.backward()
                self.optimizers['gan_d'].step()
                
                epoch_loss_d += loss_D.item()
                
                # Train Generator (5 times per discriminator update)
                for g_step in range(5):
                    self.optimizers['gan_g'].zero_grad()
                    
                    z = torch.randn(batch_size, 100).to(self.device)
                    fake_sensor_data, fake_labels = self.models['gan_generator'](z)
                    output = self.models['gan_discriminator'](fake_sensor_data, fake_labels)
                    
                    # L1 regularization for generator
                    l1_reg_g = 0
                    for param in self.models['gan_generator'].parameters():
                        l1_reg_g += torch.sum(torch.abs(param))
                    
                    loss_G = criterion(output, real_label) + 1e-5 * l1_reg_g
                    loss_G.backward()
                    self.optimizers['gan_g'].step()
                    
                    epoch_loss_g += loss_G.item()
                
                num_batches += 1
            
            avg_loss_g = epoch_loss_g / (num_batches * 5)  # 5 generator updates per batch
            avg_loss_d = epoch_loss_d / num_batches
            
            epoch_losses_g.append(avg_loss_g)
            epoch_losses_d.append(avg_loss_d)
            
            # Print every 5 epochs
            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1:3d}/{num_epochs}] - Generator Loss: {avg_loss_g:.6f}, Discriminator Loss: {avg_loss_d:.6f}")
        
        self.training_history['gan'] = {
            'epochs': list(range(1, num_epochs + 1)),
            'generator_losses': epoch_losses_g,
            'discriminator_losses': epoch_losses_d
        }

    def _train_rnn(self, num_epochs):
        """Train RNN model with regularization"""
        dataloader = DataLoader(self.rnn_dataset, batch_size=64, shuffle=True)
        criterion = nn.MSELoss()
        
        epoch_losses = []
        
        for epoch in range(num_epochs):
            epoch_loss = 0
            num_batches = 0
            
            for batch in dataloader:
                input_seq = batch['input_sequence'].to(self.device)
                target_seq = batch['target_sequence'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                self.optimizers['rnn'].zero_grad()
                output = self.models['rnn'](input_seq, labels, teacher_forcing=True)
                
                # MSE loss
                mse_loss = criterion(output, target_seq)
                
                # L1 regularization
                l1_reg = 0
                for param in self.models['rnn'].parameters():
                    l1_reg += torch.sum(torch.abs(param))
                
                # Gradient penalty for RNN stability
                grad_penalty = 0
                for param in self.models['rnn'].parameters():
                    if param.grad is not None:
                        grad_penalty += torch.sum(param.grad ** 2)
                
                total_loss = mse_loss + 1e-5 * l1_reg + 1e-6 * grad_penalty
                total_loss.backward()
                
                # Gradient clipping for RNN stability
                torch.nn.utils.clip_grad_norm_(self.models['rnn'].parameters(), max_norm=1.0)
                
                self.optimizers['rnn'].step()
                
                epoch_loss += total_loss.item()
                num_batches += 1
            
            avg_loss = epoch_loss / num_batches
            epoch_losses.append(avg_loss)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1:3d}/{num_epochs}] - Loss: {avg_loss:.6f}")
        
        self.training_history['rnn'] = {
            'epochs': list(range(1, num_epochs + 1)),
            'losses': epoch_losses
        }
    
    def _train_cgan(self, num_epochs):
        """Train cGAN model with 1:5 discriminator:generator ratio"""
        dataloader = DataLoader(self.dataset, batch_size=64, shuffle=True)
        criterion = nn.BCELoss()
        
        epoch_losses_g = []
        epoch_losses_d = []
        
        for epoch in range(num_epochs):
            epoch_loss_g = 0
            epoch_loss_d = 0
            num_batches = 0
            
            for batch_idx, batch in enumerate(dataloader):
                real_sensor_data = batch['sensor_data'].to(self.device)
                conditions = batch['labels'].to(self.device)
                batch_size = real_sensor_data.size(0)
                
                real_label = torch.ones(batch_size, 1).to(self.device) * 0.9
                fake_label = torch.zeros(batch_size, 1).to(self.device) + 0.1
                
                # Train Discriminator
                self.optimizers['cgan_d'].zero_grad()
                
                output_real = self.models['cgan_discriminator'](real_sensor_data, conditions)
                loss_D_real = criterion(output_real, real_label)
                
                z = torch.randn(batch_size, 100).to(self.device)
                fake_sensor_data = self.models['cgan_generator'](z, conditions)
                output_fake = self.models['cgan_discriminator'](fake_sensor_data.detach(), conditions)
                loss_D_fake = criterion(output_fake, fake_label)
                
                # L1 regularization
                l1_reg_d = 0
                for param in self.models['cgan_discriminator'].parameters():
                    l1_reg_d += torch.sum(torch.abs(param))
                
                loss_D = loss_D_real + loss_D_fake + 1e-5 * l1_reg_d
                loss_D.backward()
                self.optimizers['cgan_d'].step()
                
                epoch_loss_d += loss_D.item()
                
                # Train Generator (5 times)
                for g_step in range(5):
                    self.optimizers['cgan_g'].zero_grad()
                    
                    z = torch.randn(batch_size, 100).to(self.device)
                    fake_sensor_data = self.models['cgan_generator'](z, conditions)
                    output = self.models['cgan_discriminator'](fake_sensor_data, conditions)
                    
                    # L1 regularization
                    l1_reg_g = 0
                    for param in self.models['cgan_generator'].parameters():
                        l1_reg_g += torch.sum(torch.abs(param))
                    
                    loss_G = criterion(output, real_label) + 1e-5 * l1_reg_g
                    loss_G.backward()
                    self.optimizers['cgan_g'].step()
                    
                    epoch_loss_g += loss_G.item()
                
                num_batches += 1
            
            avg_loss_g = epoch_loss_g / (num_batches * 5)
            avg_loss_d = epoch_loss_d / num_batches
            
            epoch_losses_g.append(avg_loss_g)
            epoch_losses_d.append(avg_loss_d)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1:3d}/{num_epochs}] - Generator Loss: {avg_loss_g:.6f}, Discriminator Loss: {avg_loss_d:.6f}")
        
        self.training_history['cgan'] = {
            'epochs': list(range(1, num_epochs + 1)),
            'generator_losses': epoch_losses_g,
            'discriminator_losses': epoch_losses_d
        }
    
    def _train_transformer_gan(self, num_epochs):
        """Train Transformer GAN model with 1:5 discriminator:generator ratio"""
        dataloader = DataLoader(self.dataset, batch_size=64, shuffle=True)
        criterion = nn.BCELoss()
        
        epoch_losses_g = []
        epoch_losses_d = []
        
        for epoch in range(num_epochs):
            epoch_loss_g = 0
            epoch_loss_d = 0
            num_batches = 0
            
            for batch in dataloader:
                real_sensor_data = batch['sensor_data'].to(self.device)
                real_labels = batch['labels'].to(self.device)
                batch_size = real_sensor_data.size(0)
                
                real_label = torch.ones(batch_size, 1).to(self.device) * 0.9
                fake_label = torch.zeros(batch_size, 1).to(self.device) + 0.1
                
                # Train Discriminator
                self.optimizers['tgan_d'].zero_grad()
                
                output_real = self.models['tgan_discriminator'](real_sensor_data, real_labels)
                loss_D_real = criterion(output_real, real_label)
                
                z = torch.randn(batch_size, 100).to(self.device)
                fake_sensor_data, fake_labels = self.models['tgan_generator'](z)
                output_fake = self.models['tgan_discriminator'](fake_sensor_data.detach(), fake_labels.detach())
                loss_D_fake = criterion(output_fake, fake_label)
                
                # L1 regularization
                l1_reg_d = 0
                for param in self.models['tgan_discriminator'].parameters():
                    l1_reg_d += torch.sum(torch.abs(param))
                
                loss_D = loss_D_real + loss_D_fake + 1e-5 * l1_reg_d
                loss_D.backward()
                self.optimizers['tgan_d'].step()
                
                epoch_loss_d += loss_D.item()
                
                # Train Generator (5 times)
                for g_step in range(5):
                    self.optimizers['tgan_g'].zero_grad()
                    
                    z = torch.randn(batch_size, 100).to(self.device)
                    fake_sensor_data, fake_labels = self.models['tgan_generator'](z)
                    output = self.models['tgan_discriminator'](fake_sensor_data, fake_labels)
                    
                    # L1 regularization
                    l1_reg_g = 0
                    for param in self.models['tgan_generator'].parameters():
                        l1_reg_g += torch.sum(torch.abs(param))
                    
                    loss_G = criterion(output, real_label) + 1e-5 * l1_reg_g
                    loss_G.backward()
                    self.optimizers['tgan_g'].step()
                    
                    epoch_loss_g += loss_G.item()
                
                num_batches += 1
            
            avg_loss_g = epoch_loss_g / (num_batches * 5)
            avg_loss_d = epoch_loss_d / num_batches
            
            epoch_losses_g.append(avg_loss_g)
            epoch_losses_d.append(avg_loss_d)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1:3d}/{num_epochs}] - Generator Loss: {avg_loss_g:.6f}, Discriminator Loss: {avg_loss_d:.6f}")
        
        self.training_history['tgan'] = {
            'epochs': list(range(1, num_epochs + 1)),
            'generator_losses': epoch_losses_g,
            'discriminator_losses': epoch_losses_d
        }
    
    def _train_dann(self, num_epochs):
        """Train DANN model with regularization and 1:5 GAN ratio"""
        dataloader = DataLoader(self.dataset, batch_size=64, shuffle=True)
        label_criterion = nn.BCELoss()
        domain_criterion = nn.CrossEntropyLoss()
        gan_criterion = nn.BCELoss()
        
        epoch_dann_losses = []
        epoch_g_losses = []
        epoch_d_losses = []
        
        for epoch in range(num_epochs):
            p = float(epoch) / num_epochs
            lambda_ = 2. / (1. + np.exp(-10 * p)) - 1
            self.models['dann'].set_lambda(lambda_)
            
            epoch_dann_loss = 0
            epoch_g_loss = 0
            epoch_d_loss = 0
            num_batches = 0
            
            for batch in dataloader:
                sensor_data = batch['sensor_data'].to(self.device)
                labels = batch['labels'].to(self.device)
                domains = batch['domain_labels'].to(self.device)
                batch_size = sensor_data.size(0)
                
                # Train DANN
                self.optimizers['dann'].zero_grad()
                features, label_pred, domain_pred = self.models['dann'](sensor_data)
                
                label_loss = label_criterion(label_pred, labels)
                domain_loss = domain_criterion(domain_pred, domains)
                
                # L1 regularization for DANN
                l1_reg_dann = 0
                for param in self.models['dann'].parameters():
                    l1_reg_dann += torch.sum(torch.abs(param))
                
                dann_loss = label_loss + domain_loss + 1e-5 * l1_reg_dann
                dann_loss.backward(retain_graph=True)
                self.optimizers['dann'].step()
                
                epoch_dann_loss += dann_loss.item()
                
                # GAN training
                real_label = torch.ones(batch_size, 1).to(self.device) * 0.9
                fake_label = torch.zeros(batch_size, 1).to(self.device) + 0.1
                
                # Train Discriminator
                self.optimizers['dann_d'].zero_grad()
                
                output_real = self.models['dann_discriminator'](sensor_data, labels)
                loss_D_real = gan_criterion(output_real, real_label)
                
                z = torch.randn(batch_size, 100).to(self.device)
                fake_sensor_data, fake_labels = self.models['dann_generator'](z, features.detach())
                output_fake = self.models['dann_discriminator'](fake_sensor_data.detach(), fake_labels.detach())
                loss_D_fake = gan_criterion(output_fake, fake_label)
                
                # L1 regularization
                l1_reg_d = 0
                for param in self.models['dann_discriminator'].parameters():
                    l1_reg_d += torch.sum(torch.abs(param))
                
                loss_D = loss_D_real + loss_D_fake + 1e-5 * l1_reg_d
                loss_D.backward()
                self.optimizers['dann_d'].step()
                
                epoch_d_loss += loss_D.item()
                
                # Train Generator (5 times)
                for g_step in range(5):
                    self.optimizers['dann_g'].zero_grad()
                    
                    z = torch.randn(batch_size, 100).to(self.device)
                    fake_sensor_data, fake_labels = self.models['dann_generator'](z, features.detach())
                    output = self.models['dann_discriminator'](fake_sensor_data, fake_labels)
                    
                    # L1 regularization
                    l1_reg_g = 0
                    for param in self.models['dann_generator'].parameters():
                        l1_reg_g += torch.sum(torch.abs(param))
                    
                    loss_G = gan_criterion(output, real_label) + 1e-5 * l1_reg_g
                    loss_G.backward()
                    self.optimizers['dann_g'].step()
                    
                    epoch_g_loss += loss_G.item()
                
                num_batches += 1
            
            avg_dann_loss = epoch_dann_loss / num_batches
            avg_g_loss = epoch_g_loss / (num_batches * 5)
            avg_d_loss = epoch_d_loss / num_batches
            
            epoch_dann_losses.append(avg_dann_loss)
            epoch_g_losses.append(avg_g_loss)
            epoch_d_losses.append(avg_d_loss)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1:3d}/{num_epochs}] - DANN Loss: {avg_dann_loss:.6f}, Generator Loss: {avg_g_loss:.6f}, Discriminator Loss: {avg_d_loss:.6f}, Lambda: {lambda_:.4f}")
        
        self.training_history['dann'] = {
            'epochs': list(range(1, num_epochs + 1)),
            'dann_losses': epoch_dann_losses,
            'generator_losses': epoch_g_losses,
            'discriminator_losses': epoch_d_losses
        }
    
    def _print_training_summary(self):
        """Print final training summary"""
        print("\n" + "="*60)
        print("TRAINING SUMMARY")
        print("="*60)
        
        for model_name, history in self.training_history.items():
            print(f"\n{model_name.upper()} Training Results:")
            print("-" * 30)
            
            if 'generator_losses' in history:
                final_g_loss = history['generator_losses'][-1]
                final_d_loss = history['discriminator_losses'][-1]
                print(f"Final Generator Loss: {final_g_loss:.6f}")
                print(f"Final Discriminator Loss: {final_d_loss:.6f}")
                
                if 'dann_losses' in history:
                    final_dann_loss = history['dann_losses'][-1]
                    print(f"Final DANN Loss: {final_dann_loss:.6f}")
            else:
                final_loss = history['losses'][-1]
                print(f"Final Loss: {final_loss:.6f}")
        
        print("\n" + "="*60)
        print("All models trained successfully!")
        print("="*60)
    
    def save_all_models(self):
        """Save all trained models"""
        print("\nSaving all models...")
        for model_name, model in self.models.items():
            torch.save(model.state_dict(), os.path.join(self.save_dir, f'{model_name}.pth'))
        
        # Save training history
        with open(os.path.join(self.save_dir, 'training_history.pickle'), 'wb') as f:
            pickle.dump(self.training_history, f)
        
        print(f"All models and training history saved to {self.save_dir}")
    
    def load_all_models(self):
        """Load all saved models"""
        print("Loading all models...")
        for model_name, model in self.models.items():
            model_path = os.path.join(self.save_dir, f'{model_name}.pth')
            if os.path.exists(model_path):
                model.load_state_dict(torch.load(model_path, map_location=self.device))
                print(f"Loaded {model_name}")
            else:
                print(f"Warning: {model_name} not found")
        
        # Load training history
        history_path = os.path.join(self.save_dir, 'training_history.pickle')
        if os.path.exists(history_path):
            with open(history_path, 'rb') as f:
                self.training_history = pickle.load(f)
            print("Training history loaded")   

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize trainer
trainer = UnifiedModelTrainer(train_df, device=device)
trainer.initialize_models()

# Train all models
epochs_dict = {
    'gan': 100,
    'rnn': 100,
    'cgan': 100,
    'tgan': 100,
    'dann': 100
}

trainer.train_all_models(epochs_dict)

# Generating and saving data

In [ ]:
class DataGenerator:
    def __init__(self, trainer, device='cuda'):
        self.trainer = trainer
        self.device = device
        self.generated_data = {}
    
    def generate_all_data(self):
        """Generate data from all trained models"""
        print("Starting data generation...")
        
        # GAN: Generate 10000 samples
        print("Generating GAN data...")
        self.generated_data['gan'] = self._generate_gan_data(10000)
        
        # cGAN: Generate 1000 samples for each label combination
        print("Generating cGAN data...")
        self.generated_data['cgan'] = self._generate_cgan_data()
        
        # RNN: Generate sequences from first timestamps
        print("Generating RNN data...")
        self.generated_data['rnn'] = self._generate_rnn_data()
        
        # Transformer GAN: Generate 10000 samples
        print("Generating Transformer GAN data...")
        self.generated_data['tgan'] = self._generate_transformer_gan_data(10000)
        
        # DANN: Generate 10000 samples
        print("Generating DANN data...")
        self.generated_data['dann'] = self._generate_dann_data(10000)
        
        # Convert binary labels to original labels for all generated data
        self._convert_binary_to_original_labels()
        
        return self.generated_data
    
    def _convert_binary_to_original_labels(self):
        """Convert binary labels (label_1~4) back to original labels for all generated data"""
        print("Converting binary labels to original labels...")
        
        for model_name, data in self.generated_data.items():
            if 'labels' in data:
                # Convert binary labels to original labels
                binary_labels = data['labels']  # Shape: (n_samples, 4)
                original_labels = binary_to_original_label(binary_labels)
                
                # Add original labels to the data
                data['original_labels'] = np.array(original_labels)
                
                print(f"  {model_name}: Converted {len(original_labels)} binary labels to original labels")
    
    def _generate_gan_data(self, num_samples):
        """Generate data from GAN"""
        self.trainer.models['gan_generator'].eval()
        
        synthetic_sensor_data = []
        synthetic_labels = []
        
        with torch.no_grad():
            for i in range(0, num_samples, 100):
                batch_size = min(100, num_samples - i)
                z = torch.randn(batch_size, 100).to(self.device)
                sensor_data, labels = self.trainer.models['gan_generator'](z)
                
                synthetic_sensor_data.append(sensor_data.cpu().numpy())
                synthetic_labels.append(labels.cpu().numpy())
        
        return {
            'sensor_data': np.concatenate(synthetic_sensor_data, axis=0),
            'labels': np.concatenate(synthetic_labels, axis=0)
        }
    
    def _generate_cgan_data(self):
        """Generate data from cGAN for each label combination"""
        self.trainer.models['cgan_generator'].eval()
        
        # Generate all possible label combinations (2^4 = 16 combinations)
        label_combinations = []
        for i in range(16):
            binary = format(i, '04b')
            label_combo = [int(b) for b in binary]
            label_combinations.append(label_combo)
        
        all_sensor_data = []
        all_labels = []
        
        print(f"  Generating data for {len(label_combinations)} label combinations...")
        
        with torch.no_grad():
            for idx, label_combo in enumerate(label_combinations):
                print(f"    Generating for label combination {idx+1}/{len(label_combinations)}: {label_combo}")
                conditions = torch.FloatTensor([label_combo] * 1000).to(self.device)
                
                for i in range(0, 1000, 100):
                    batch_size = min(100, 1000 - i)
                    z = torch.randn(batch_size, 100).to(self.device)
                    sensor_data = self.trainer.models['cgan_generator'](z, conditions[i:i+batch_size])
                    
                    all_sensor_data.append(sensor_data.cpu().numpy())
                    all_labels.append(conditions[i:i+batch_size].cpu().numpy())
        
        return {
            'sensor_data': np.concatenate(all_sensor_data, axis=0),
            'labels': np.concatenate(all_labels, axis=0)
        }
    
    def _generate_rnn_data(self):
        """Generate RNN sequences from first timestamps"""
        self.trainer.models['rnn'].eval()
        
        all_generated_sequences = []
        all_labels = []
        
        print(f"  Generating sequences for {len(self.trainer.dataset)} samples...")
        
        with torch.no_grad():
            for idx in range(len(self.trainer.dataset)):
                if idx % 1000 == 0:
                    print(f"    Processing sample {idx+1}/{len(self.trainer.dataset)}")
                
                batch = self.trainer.dataset[idx]
                initial_sensor = batch['sensor_data'][0:1, :].to(self.device)  # First timestamp
                labels = batch['labels'].unsqueeze(0).to(self.device)
                
                # Generate 127 more timestamps (128 total - 1 initial = 127)
                generated_sequence = self.trainer.models['rnn'].generate_sequence(
                    initial_sensor, labels, seq_len=127
                )
                
                # Combine initial + generated
                full_sequence = torch.cat([initial_sensor.unsqueeze(0), generated_sequence], dim=1)
                
                all_generated_sequences.append(full_sequence.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        
        return {
            'sensor_data': np.concatenate(all_generated_sequences, axis=0),
            'labels': np.concatenate(all_labels, axis=0)
        }
    
    def _generate_transformer_gan_data(self, num_samples):
        """Generate data from Transformer GAN"""
        self.trainer.models['tgan_generator'].eval()
        
        synthetic_sensor_data = []
        synthetic_labels = []
        
        with torch.no_grad():
            for i in range(0, num_samples, 100):
                batch_size = min(100, num_samples - i)
                z = torch.randn(batch_size, 100).to(self.device)
                sensor_data, labels = self.trainer.models['tgan_generator'](z)
                
                synthetic_sensor_data.append(sensor_data.cpu().numpy())
                synthetic_labels.append(labels.cpu().numpy())
        
        return {
            'sensor_data': np.concatenate(synthetic_sensor_data, axis=0),
            'labels': np.concatenate(synthetic_labels, axis=0)
        }
    
    def _generate_dann_data(self, num_samples):
        """Generate data from DANN"""
        self.trainer.models['dann'].eval()
        self.trainer.models['dann_generator'].eval()
        
        # Extract features from real data
        dataloader = DataLoader(self.trainer.dataset, batch_size=64, shuffle=False)
        all_features = []
        
        print("  Extracting domain-invariant features from real data...")
        with torch.no_grad():
            for batch in dataloader:
                sensor_data = batch['sensor_data'].to(self.device)
                features, _, _ = self.trainer.models['dann'](sensor_data)
                all_features.append(features)
        
        all_features = torch.cat(all_features, dim=0)
        
        # Sample features for generation
        indices = torch.randint(0, all_features.size(0), (num_samples,))
        sampled_features = all_features[indices]
        
        synthetic_sensor_data = []
        synthetic_labels = []
        
        print("  Generating domain-invariant synthetic data...")
        with torch.no_grad():
            for i in range(0, num_samples, 100):
                batch_size = min(100, num_samples - i)
                z = torch.randn(batch_size, 100).to(self.device)
                features_batch = sampled_features[i:i+batch_size]
                
                sensor_data, labels = self.trainer.models['dann_generator'](z, features_batch)
                
                synthetic_sensor_data.append(sensor_data.cpu().numpy())
                synthetic_labels.append(labels.cpu().numpy())
        
        return {
            'sensor_data': np.concatenate(synthetic_sensor_data, axis=0),
            'labels': np.concatenate(synthetic_labels, axis=0)
        }
    
    def save_generated_data(self, save_dir='../../results/generated_data'):
        """Save all generated data with both binary and original labels"""
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"Saving generated data to {save_dir}...")
        
        for model_name, data in self.generated_data.items():
            print(f"  Saving {model_name} data...")
            
            # Prepare data for DataFrame
            df_data = {
                'sensor_data': list(data['sensor_data']),
                'label': data['original_labels'],  # Original labels (0-11)
                'label_1': data['labels'][:, 0],   # Binary label 1
                'label_2': data['labels'][:, 1],   # Binary label 2
                'label_3': data['labels'][:, 2],   # Binary label 3
                'label_4': data['labels'][:, 3]    # Binary label 4
            }
            
            # Create DataFrame
            df = pd.DataFrame(df_data)
            
            # Save as pickle
            pickle_path = os.path.join(save_dir, f'{model_name}_generated_data.pickle')
            with open(pickle_path, 'wb') as f:
                pickle.dump(df.to_dict('records'), f)
            
            # Save as CSV for easy inspection
            csv_path = os.path.join(save_dir, f'{model_name}_generated_data.csv')
            df_summary = df.drop('sensor_data', axis=1)  # Remove sensor_data for CSV (too large)
            df_summary.to_csv(csv_path, index=False)
            
            print(f"    Saved {len(df)} samples")
            print(f"    Pickle: {pickle_path}")
            print(f"    CSV summary: {csv_path}")
            
            # Print label distribution
            label_counts = df['label'].value_counts().sort_index()
            print(f"    Label distribution:")
            for label, count in label_counts.items():
                print(f"      Label {label}: {count} samples")
    
    def get_generation_summary(self):
        """Get summary of generated data"""
        summary = {}
        
        for model_name, data in self.generated_data.items():
            if 'original_labels' in data:
                unique_labels, counts = np.unique(data['original_labels'], return_counts=True)
                label_distribution = dict(zip(unique_labels, counts))
                
                summary[model_name] = {
                    'total_samples': len(data['sensor_data']),
                    'sensor_data_shape': data['sensor_data'].shape,
                    'label_distribution': label_distribution,
                    'unique_labels': len(unique_labels)
                }
        
        return summary
    
    def print_generation_summary(self):
        """Print detailed summary of generated data"""
        print("\n" + "="*60)
        print("GENERATED DATA SUMMARY")
        print("="*60)
        
        summary = self.get_generation_summary()
        
        for model_name, info in summary.items():
            print(f"\n{model_name.upper()}:")
            print(f"  Total samples: {info['total_samples']:,}")
            print(f"  Sensor data shape: {info['sensor_data_shape']}")
            print(f"  Unique labels: {info['unique_labels']}")
            print(f"  Label distribution:")
            
            for label in sorted(info['label_distribution'].keys()):
                count = info['label_distribution'][label]
                percentage = (count / info['total_samples']) * 100
                print(f"    Label {label}: {count:,} samples ({percentage:.1f}%)")
        
        print("\n" + "="*60)

# Generate data
generator = DataGenerator(trainer, device=device)
generated_data = generator.generate_all_data()
generator.save_generated_data()

print("Training and data generation completed!")
print("Generated data summary:")
for model_name, data in generated_data.items():
    print(f"  {model_name}: {data['sensor_data'].shape[0]} samples")
